In [0]:
from pyspark.sql.functions import col, when

# Read Bronze Table
bronze_df = spark.table(
    "workspace.default.capstone_bronze_sales"
)

# Create Quality Flag
dq_df = bronze_df.withColumn(
    "quality_flag",
    when(col("customer_id").isNull(), "INVALID_CUSTOMER")
    .when(col("quantity") <= 0, "INVALID_QUANTITY")
    .when(col("net_amount") < 0, "INVALID_AMOUNT")
    .when(col("product_id") == "UNKNOWN", "INVALID_PRODUCT")
    .otherwise("VALID")
)

# Create temporary view
dq_df.createOrReplaceTempView(
    "capstone_sales_quality"
)

# Display quality summary
display(
    dq_df.groupBy("quality_flag").count()
)
dq_df.write.mode("overwrite").saveAsTable(
    "workspace.default.capstone_sales_quality"
)

In [0]:
%sql
SELECT quality_flag,
       COUNT(*)
FROM workspace.default.capstone_sales_quality
GROUP BY quality_flag;